# Pipeline de Análise, Visualização e Machine Learning (Scikit-Learn)
## Projeto: xApp RDL (Resource and Decision Layer) — Fase 1 (H-RDL)

Este notebook executa no **Google Colab** para:
1. Ingerir os datasets CSV gerados pela co-simulação (`dataset_flow_metrics.csv` e `dataset_rdl_decisions_ml.csv`).
2. Gerar visualizações estatísticas (CDF de latência, boxplots por fatia de rede, taxas de colisão).
3. Treinar e avaliar algoritmos de classificação do **Scikit-Learn** (`RandomForestClassifier`, `DecisionTreeClassifier`, `GradientBoostingClassifier`).
4. Gerar relatórios automatizados de conformidade e desempenho.

In [ ]:
# 1. Instalação e Importação de Bibliotecas Essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
print("Bibliotecas carregadas com sucesso!")

In [ ]:
# 2. Carregamento dos Datasets CSV
# Se estiver rodando no Google Colab, faça o upload dos arquivos ou use os caminhos locais:
try:
    df_flows = pd.read_csv('experiments/results/dataset_flow_metrics.csv')
    df_ml = pd.read_csv('experiments/results/dataset_rdl_decisions_ml.csv')
    print("Datasets carregados localmente!")
except Exception:
    print("Faça o upload manual no Colab ou aponte para o caminho correto.")
    # from google.colab import files
    # uploaded = files.upload()
    # df_flows = pd.read_csv('dataset_flow_metrics.csv')
    # df_ml = pd.read_csv('dataset_rdl_decisions_ml.csv')

print("Shape dos Fluxos:", df_flows.shape)
print("Shape das Decisões/Telemetria:", df_ml.shape)
df_ml.head()

In [ ]:
# 3. Visualização Estatística e Gráficos Comparativos
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Boxplot de Latência por Fatia e Cenário
sns.boxplot(data=df_flows, x='slice_type', y='mean_delay_ms', hue='scenario', palette='Set2', ax=axes[0])
axes[0].axhline(5.0, color='red', linestyle='--', label='SLA URLLC (5 ms)')
axes[0].set_title('Distribuição de Latência Fim-a-Fim por Fatia')
axes[0].set_ylabel('Latência Média (ms)')
axes[0].legend()

# Taxa de Entrega de Pacotes (PDR)
sns.barplot(data=df_flows, x='slice_type', y='delivery_ratio_pct', hue='scenario', palette='Blues_d', ax=axes[1])
axes[1].set_title('Taxa de Entrega de Pacotes (PDR %)')
axes[1].set_ylabel('PDR (%)')

plt.tight_layout()
plt.show()

In [ ]:
# 4. Machine Learning com Scikit-Learn: Classificação de Conflitos de Ação O-RAN
# Objetivo: Predizer se um estado de rádio resultará em conflito (conflict_flag: 0 ou 1)

features = ['ue_count', 'traffic_load_mbps', 'rsrp_dbm', 'sinr_db', 'prb_demanded', 'tx_power_dbm']
X = df_ml[features]
y = df_ml['conflict_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Treinamento do Modelo Random Forest
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("=== Relatório de Classificação (Scikit-Learn Random Forest) ===")
print(classification_report(y_test, y_pred, target_names=['Sem Conflito', 'Conflito Detectado']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# 5. Matriz de Confusão e Importância de Atributos (Feature Importance)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Normal', 'Conflito'], yticklabels=['Normal', 'Conflito'], ax=axes[0])
axes[0].set_title('Matriz de Confusão - Detecção de Conflitos')
axes[0].set_xlabel('Predição')
axes[0].set_ylabel('Real')

# Feature Importance
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
importances.plot(kind='barh', color='teal', ax=axes[1])
axes[1].set_title('Importância das Variáveis de Rádio na Decisão da RDL')
axes[1].set_xlabel('Importância Relativa (Gini)')

plt.tight_layout()
plt.show()